<a href="https://colab.research.google.com/github/175329/Chatbot-IOT/blob/main/chatbot_actualizado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install ipywidgets --quiet
from google.colab import output
output.enable_custom_widget_manager()

import re
import nltk
from collections import defaultdict
from IPython.display import display, HTML
import ipywidgets as widgets

nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from google.colab import drive
drive.mount('/content/drive')

CORPUS_PATH = "/content/drive/MyDrive/upslp/corpus_tecnologia.txt"

def cargar_corpus(path):
    corpus = {}
    tema_actual = None
    try:
        with open(path, "r", encoding="utf-8") as f:
            for linea in f:
                linea = linea.strip()
                if not linea:
                    continue
                if linea.startswith("[") and linea.endswith("]"):
                    tema_actual = linea[1:-1].lower()
                    corpus[tema_actual] = []
                elif tema_actual:
                    corpus[tema_actual].append(linea)
        total = sum(len(v) for v in corpus.values())
        print(f"✓ Corpus cargado: {total} oraciones en {len(corpus)} temas")
        for tema, oraciones in corpus.items():
            print(f"  [{tema}] → {len(oraciones)} oraciones")
    except FileNotFoundError:
        print("⚠ No se encontró el archivo del corpus")
    return corpus

CORPUS = cargar_corpus(CORPUS_PATH)

PALABRAS_CLAVE = {
    "virus": [
        "virus", "malware", "troyano", "ransomware", "spyware",
        "antivirus", "gusano", "infección", "infectar", "adware",
        "rootkit", "botnet", "keylogger", "exploit"
    ],
    "ciberseguridad": [
        "ciberseguridad", "seguridad", "ataque", "phishing", "firewall",
        "cifrado", "ddos", "vpn", "contraseña", "hack", "vulnerabilidad",
        "pentesting", "brecha", "intrusión", "parche", "siem", "zero trust"
    ],
    "inteligencia artificial": [
        "inteligencia", "artificial", "ia", "machine", "learning", "deep",
        "neural", "neuronal", "gpt", "chatbot", "nlp", "transformer",
        "generativa", "entrenamiento", "modelo", "algoritmo"
    ],
    "iot": [
        "iot", "internet", "cosas", "sensor", "dispositivo", "wearable",
        "smart", "inteligente", "zigbee", "mqtt", "edge", "gateway",
        "industria", "conectado", "actuador"
    ],
    "redes": [
        "red", "redes", "tcp", "ip", "router", "switch", "wifi", "dns",
        "protocolo", "5g", "lan", "wan", "subred", "vlan", "qos",
        "https", "sdn", "latencia", "ancho", "banda"
    ],
    "computacion en la nube": [
        "nube", "cloud", "aws", "azure", "google cloud", "contenedor",
        "docker", "kubernetes", "serverless", "cdn", "saas", "paas",
        "iaas", "elasticidad", "migración", "híbrida"
    ],
    "computacion cuantica": [
        "cuántica", "cuantica", "qubit", "quantum", "superposición",
        "entrelazamiento", "ibm", "grover", "shor", "criptografía"
    ],
    "programacion": [
        "programación", "programacion", "python", "código", "codigo",
        "algoritmo", "api", "git", "desarrollo", "software", "objeto",
        "clase", "test", "microservicio", "docker", "agil", "sprint"
    ],
}

# Set global de todas las palabras clave para filtro de tecnología
TODAS_LAS_PALABRAS = {p for lista in PALABRAS_CLAVE.values() for p in lista}

# ============================================================
# CELDA 5 — Motor de búsqueda por similitud
# ============================================================
STOPWORDS_ES = set(stopwords.words('spanish'))

def limpiar(texto):
    texto = texto.lower()
    texto = re.sub(r'[^\w\s]', '', texto)
    return [t for t in texto.split() if t not in STOPWORDS_ES and len(t) > 2]

def detectar_tema(tokens):
    scores = defaultdict(int)
    for token in tokens:
        for tema, palabras in PALABRAS_CLAVE.items():
            for palabra in palabras:
                if token == palabra:
                    scores[tema] += 3
                elif token in palabra or palabra in token:
                    scores[tema] += 1
    return max(scores, key=scores.get) if scores else None

def similitud_jaccard(tokens_a, tokens_b):
    a, b = set(tokens_a), set(tokens_b)
    if not a or not b:
        return 0
    return len(a & b) / len(a | b)

def buscar_oraciones(tokens_input, tema_detectado, top_n=3):
    candidatas = []

    for tema, oraciones in CORPUS.items():
        bonus = 0.25 if tema == tema_detectado else 0
        for oracion in oraciones:
            score = similitud_jaccard(tokens_input, limpiar(oracion)) + bonus
            if score > 0:
                candidatas.append((score, oracion))

    vistas = set()
    resultado = []
    for score, oracion in sorted(candidatas, reverse=True):
        if oracion not in vistas:
            vistas.add(oracion)
            resultado.append(oracion)
        if len(resultado) >= top_n:
            break
    return resultado

def es_tecnologia(tokens):
    return any(t in TODAS_LAS_PALABRAS for t in tokens)

def generar_respuesta(input_usuario):
    tokens = limpiar(input_usuario)

    if not es_tecnologia(tokens):
        return ("Lo siento, solo respondo sobre tecnología: IA, IoT, "
                "ciberseguridad, redes, computación y temas relacionados.")

    tema = detectar_tema(tokens)
    oraciones = buscar_oraciones(tokens, tema, top_n=3)

    if not oraciones:
        return ("No encontré información específica. Intenta preguntar sobre: "
                "IA, virus, redes, IoT, nube, ciberseguridad o computación cuántica.")

    encabezado = f"Sobre {tema.upper()}:\n\n" if tema else ""
    return encabezado + "\n\n".join(f"• {o}" for o in oraciones)

# ============================================================
# CELDA 6 — Interfaz gráfica
# ============================================================
CSS = """
<style>
#chat-container {
    max-width: 680px; margin: 20px auto;
    font-family: 'Segoe UI', Arial, sans-serif;
    border-radius: 12px; overflow: hidden;
    border: 1px solid #dde3f0;
    box-shadow: 0 2px 12px rgba(0,0,0,0.08);
}
#chat-header {
    background: #1a1a2e; color: #e0e0ff;
    padding: 14px 20px; display: flex;
    align-items: center; gap: 10px;
}
.header-dot { width:10px; height:10px; border-radius:50%; background:#4ade80; }
.header-title { font-size:15px; font-weight:600; }
.header-sub { font-size:12px; color:#9999cc; margin-left:auto; }
#chat-log {
    background: #f7f8fc; min-height: 400px; max-height: 520px;
    overflow-y: auto; padding: 16px;
    display: flex; flex-direction: column; gap: 12px;
}
.msg-row { display:flex; gap:10px; align-items:flex-start; }
.msg-row.user { flex-direction:row-reverse; }
.avatar {
    width:34px; height:34px; border-radius:50%;
    display:flex; align-items:center; justify-content:center;
    font-size:12px; font-weight:600; flex-shrink:0;
}
.av-bot { background:#e6eeff; color:#1a1a2e; }
.av-user { background:#2d2d50; color:#ccceff; }
.bubble { max-width:75%; padding:10px 14px; font-size:14px; line-height:1.7; white-space:pre-wrap; }
.bubble.bot {
    background:#ffffff; border:1px solid #dde3f0; color:#222;
    border-radius:4px 14px 14px 14px;
}
.bubble.user {
    background:#1a1a2e; color:#e8e8ff;
    border-radius:14px 4px 14px 14px;
}
</style>
"""

display(HTML(CSS))

chat_output = widgets.Output()
text_input  = widgets.Text(
    placeholder='Pregunta sobre IA, virus, IoT, redes, nube...',
    layout=widgets.Layout(width='85%', height='38px')
)
send_button = widgets.Button(
    description='Enviar',
    layout=widgets.Layout(width='13%', height='38px')
)
send_button.style.button_color = '#1a1a2e'

HEADER_HTML = """
<div id="chat-header">
  <div class="header-dot"></div>
  <span class="header-title">TechBot</span>
  <span class="header-sub">IA · Redes · IoT · Ciberseguridad</span>
</div>"""

def render_msg(texto, rol):
    av  = "av-bot" if rol == "bot" else "av-user"
    bub = "bot"    if rol == "bot" else "user"
    lbl = "TB"     if rol == "bot" else "TÚ"
    return f'<div class="msg-row {rol}"><div class="avatar {av}">{lbl}</div><div class="bubble {bub}">{texto}</div></div>'

historial = [render_msg(
    "Hola, soy TechBot. Pregúntame sobre IA, virus, IoT, redes, "
    "ciberseguridad, computación cuántica, nube o programación.", "bot"
)]

def actualizar_chat():
    with chat_output:
        chat_output.clear_output(wait=True)
        display(HTML(
            '<div id="chat-container">'
            + HEADER_HTML
            + '<div id="chat-log">'
            + ''.join(historial)
            + '</div></div>'
        ))

def enviar(b=None):
    pregunta = text_input.value.strip()
    if not pregunta:
        return
    text_input.value = ''
    historial.append(render_msg(pregunta, "user"))
    respuesta = generar_respuesta(pregunta).replace('\n', '<br>')
    historial.append(render_msg(respuesta, "bot"))
    actualizar_chat()

send_button.on_click(enviar)
text_input.on_submit(enviar)

display(chat_output)
display(widgets.HBox(
    [text_input, send_button],
    layout=widgets.Layout(max_width='680px', margin='0 auto', gap='8px')
))
actualizar_chat()

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 324, in run
    session = self.get_default_session(options)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/index_command.py", line 71, in get_default_session
    self._session = self.enter_context(self._build_session(options))
                                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/index_command.py", line 100, in _build_session
    session = PipSession(
              ^^^^^^^^^

Output()